In [1]:
# Section 1
from dotenv import load_dotenv
import langchain
from langchain_ollama import ChatOllama

load_dotenv()
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from langchain_core.tools import tool

# Reusing the familiar tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28°C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "464.00 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

print("Tools ready: get_weather, get_stock_price")



Tools ready: get_weather, get_stock_price


In [3]:
from langchain.agents import create_agent

agent_no_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful AI assistant"
)

In [8]:
# We can reuse the same agent — memory is not in the agent, it's in how we call it
conversation_history = []  # We manage this list
def chat(user_message: str) -> str:
    """Send a message and keep the full history for next time."""
    # Add the new user message to history
    conversation_history.append({"role": "user", "content": user_message})
    
    # Invoke with the FULL history every time
    #response = agent_no_memory.invoke({"messages": [{"role": "user", "content": user_message}]})
    response = agent_no_memory.invoke({"messages": conversation_history})

    # Extract the AI reply and store it too
    ai_reply = response["messages"][-1].content
    conversation_history.append({"role": "assistant", "content": ai_reply})
    
    return ai_reply

print("Manual chat() function ready.")

Manual chat() function ready.


In [9]:
print('Turn 1')
response_1 = chat('My name is Anand and I work with Zensar Technologies Pune')
print(f'Agent: {response_1}\n')

print('Turn 2')
response_2 = chat('What is the weather in Pune today?')
print(f'Agent: {response_2}\n')

print('Turn 3')
response_3 = chat('What is my name and where do I work?')
print(f'Agent: {response_3}\n')

Turn 1
Agent: Nice to meet you, Anand! 😊 It’s great to hear you’re with Zensor Technologies in Pune. How can I assist you today? Whether you need information about the company, market data, tech resources, or anything else, just let me know!

Turn 2
Agent: Great news—Pune is enjoying sunny skies today with a high around **28 °C**. If you’re heading out, it should be a pleasant day for work or a quick break! Let me know if you need anything else (e.g., an outlook for the rest of the week, travel tips, or info about Zensar).

Turn 3
Agent: Your name is **Anand**, and you work with **Zensar Technologies in Pune**.



In [10]:
from langgraph.checkpoint.memory import MemorySaver

memory_saver = MemorySaver()

agent_with_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful AI assistant",
    checkpointer=memory_saver
)
print('Agent with memory created!!!')

Agent with memory created!!!


In [13]:
config = {"configurable": {"thread_id": "South_indian_dishes_receipes_anand"}}

print('Turn 1')
response_1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "My name is Anand and I work with Zensar Technologies Pune"}]},
    config=config
)
print(f'Agent: {response_1['messages'][-1].content}\n')

print('Turn 2')
response_2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Pune today?"}]},
    config=config
)
print(f'Agent: {response_2['messages'][-1].content}\n')

print('Turn 3')
response_3 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is my name and where do I work?"}]},
    config=config
)
print(f'Agent: {response_3['messages'][-1].content}\n')

Turn 1
Agent: Got it, Anand! You work at **Zensar Technologies** in **Pune**. Let me know if there’s anything else you’d like assistance with.

Turn 2
Agent: Today in Pune it’s **sunny**, with a high around **28 °C** and a low near **18 °C**. Light breezes are expected and there’s no rain in the forecast. Let me know if you’d like a more detailed forecast, hourly updates, or anything else!

Turn 3
Agent: Your name is **Anand**, and you work at **Zensar Technologies** in **Pune**.



In [14]:
config_tom = {"configurable": {"thread_id": "South_indian_dishes_receipes_tom"}}
config_jerry = {"configurable": {"thread_id": "South_indian_dishes_receipes_jerry"}}

print('Turn 1_1')
response_1_1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "My name is Tom and I like cricket"}]},
    config=config_tom
)
print(f'Agent: {response_1_1['messages'][-1].content}\n')

print('Turn 1_2')
response_1_2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "My name is Jerry and I like music"}]},
    config=config_jerry
)
print(f'Agent: {response_1_2['messages'][-1].content}\n')


print('Turn 2_1')
response_2_1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Pune today?"}]},
    config=config_tom
)
print(f'Agent: {response_2_1['messages'][-1].content}\n')

print('Turn 2_2')
response_2_2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Pune today?"}]},
    config=config_jerry
)
print(f'Agent: {response_2_2['messages'][-1].content}\n')


print('Turn 3_1')
response_3_1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is my name and what is my hobby?"}]},
    config=config_tom
)
print(f'Agent: {response_3_1['messages'][-1].content}\n')

print('Turn 3_2')
response_3_2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What is my name and what is my hobby?"}]},
    config=config_jerry
)
print(f'Agent: {response_3_2['messages'][-1].content}\n')


Turn 1_1
Agent: Nice to meet you, Tom! Cricket is a great sport—whether you enjoy watching the games, playing, or following the stats, there’s always something exciting happening on the field. Do you have a favorite team or player?

Turn 1_2
Agent: Nice to meet you, Jerry! 🎶  
What kind of music do you enjoy? Do you have favorite artists, genres, or playlists you’re into right now? I’d love to hear about your musical taste—and if you’d like, I can suggest some new songs or artists to check out!

Turn 2_1
Agent: The current weather in Pune is sunny with a high of around 28 °C. Let me know if you’d like a longer forecast or any other details!

Turn 2_2
Agent: The weather in Pune today is **sunny** with a **high of around 28 °C**. Let me know if you’d like a more detailed forecast (e.g., chance of rain, humidity, or an outlook for the rest of the week)!

Turn 3_1
Agent: You mentioned that your name is **Tom** and that you enjoy **cricket** as your hobby.

Turn 3_2
Agent: You told me earli

In [16]:
# Retrieve thread specific conversation state
jerry_state = memory_saver.get(config_jerry)
print(jerry_state)

tom_state = memory_saver.get(config_tom)
print(tom_state)


{'v': 4, 'ts': '2026-07-24T04:46:43.036948+00:00', 'id': '1f1871aa-ad7d-6ecb-8009-4a0922a1eb17', 'channel_versions': {'__start__': '00000000000000000000000000000010.0.5549517276710076', 'messages': '00000000000000000000000000000011.0.7808939775656587', 'branch:to:model': '00000000000000000000000000000011.0.7808939775656587', '__pregel_tasks': '00000000000000000000000000000007.0.6432764330504431'}, 'versions_seen': {'__input__': {}, '__start__': {'__start__': '00000000000000000000000000000009.0.5636470394587134'}, 'model': {'branch:to:model': '00000000000000000000000000000010.0.5549517276710076'}, 'tools': {}}, 'updated_channels': ['messages'], 'channel_values': {'messages': [HumanMessage(content='My name is Jerry and I like music', additional_kwargs={}, response_metadata={}, id='e02e311f-f0aa-4962-966a-3fb119ffc551'), AIMessage(content='Nice to meet you, Jerry! 🎶  \nWhat kind of music do you enjoy? Do you have favorite artists, genres, or playlists you’re into right now? I’d love to he

In [20]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "conversation.receipes"
conn = sqlite3.connect(db_path, check_same_thread=False)
sqllite_saver = SqliteSaver(conn)

agent_with_sqlite = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful AI assistant",
    checkpointer=sqllite_saver
)
print('Agent with Sqlite created!!!')


Agent with Sqlite created!!!


In [21]:
config_tom = {"configurable": {"thread_id": "South_indian_dishes_receipes_tom"}}
config_jerry = {"configurable": {"thread_id": "South_indian_dishes_receipes_jerry"}}

print('Turn 1_1')
response_1_1 = agent_with_sqlite.invoke(
    {"messages": [{"role": "user", "content": "My name is Tom and I like cricket"}]},
    config=config_tom
)
print(f'Agent: {response_1_1['messages'][-1].content}\n')

print('Turn 1_2')
response_1_2 = agent_with_sqlite.invoke(
    {"messages": [{"role": "user", "content": "My name is Jerry and I like music"}]},
    config=config_jerry
)
print(f'Agent: {response_1_2['messages'][-1].content}\n')


print('Turn 2_1')
response_2_1 = agent_with_sqlite.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Pune today?"}]},
    config=config_tom
)
print(f'Agent: {response_2_1['messages'][-1].content}\n')

print('Turn 2_2')
response_2_2 = agent_with_sqlite.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Pune today?"}]},
    config=config_jerry
)
print(f'Agent: {response_2_2['messages'][-1].content}\n')


print('Turn 3_1')
response_3_1 = agent_with_sqlite.invoke(
    {"messages": [{"role": "user", "content": "What is my name and what is my hobby?"}]},
    config=config_tom
)
print(f'Agent: {response_3_1['messages'][-1].content}\n')

print('Turn 3_2')
response_3_2 = agent_with_sqlite.invoke(
    {"messages": [{"role": "user", "content": "What is my name and what is my hobby?"}]},
    config=config_jerry
)
print(f'Agent: {response_3_2['messages'][-1].content}\n')


Turn 1_1
Agent: Nice to meet you, Tom! 🎉 Cricket is such an exciting sport—do you have a favorite team or player? Or maybe there’s a particular match or tournament you’ve been following lately? Let me know what you’d like to chat about!

Turn 1_2
Agent: Nice to meet you, Jerry! 🎵  
Music is such a wonderful passion—there’s always something new to discover. Do you have any favorite genres, artists, or songs that you keep on repeat? If you’re looking for recommendations or want to chat about the latest releases, just let me know!

Turn 2_1
Agent: Great! If you need any more details—like a forecast for the rest of the week, or tips on staying cool while watching a cricket match—just let me know!

Turn 2_2
Agent: The current weather in Pune is sunny with a high around 28 °C (about 82 °F). If you need more details—like the temperature right now, humidity, or a forecast for the next few days—just let me know!

Turn 3_1
Agent: Your name is **Tom**, and you enjoy **cricket** as your hobby.

Tu